# TruLens RAG Evaluation
Evaluate your RAG system using the **RAG Triad**:
- **Answer Relevance** – Is the response relevant to the query?
- **Context Relevance** – Is the retrieved context relevant to the query?
- **Groundedness** – Is the response grounded in the retrieved context?

## 1. Install dependencies

In [ ]:
!pip install trulens-providers-openai langchain langchain-openai langchain-community faiss-cpu python-dotenv -q

## 2. Load environment variables

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
print("API key loaded:", bool(openai_api_key))

## 3. Set up the RAG pipeline

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

def build_retriever(kb_id: str, k: int = 5):
    docs_dir = Path("data") / kb_id / "documents"
    documents = []
    for f in docs_dir.glob("*.txt"):
        documents.extend(TextLoader(str(f), encoding="utf-8").load())

    chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(documents)
    embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)
    vector_store = FAISS.from_documents(chunks, embeddings)
    return vector_store.as_retriever(search_type="similarity", search_kwargs={"k": k})

print("RAG pipeline ready")

## 4. Run a query and collect context + response

In [ ]:
from langchain_openai import OpenAI

KB_ID = "kb_example"  # replace with your actual KB id
QUERY = "What do polar bears eat?"

retriever = build_retriever(KB_ID)
llm = OpenAI(openai_api_key=openai_api_key)

retrieved_docs = retriever.invoke(QUERY)
contexts = [doc.page_content for doc in retrieved_docs]
context_str = "\n".join(contexts)

prompt = [f"Use the following information to answer the question:\n\n{context_str}\n\nQuestion: {QUERY}"]
result = llm.generate(prompt)
response = result.generations[0][0].text.strip()

print("Response:", response)

## 5. Initialise the TruLens OpenAI provider

In [ ]:
from trulens.providers.openai import OpenAI as OpenAIProvider

provider = OpenAIProvider()
print("TruLens provider ready")

## 6. Answer Relevance
Is the final response relevant to the original query?

In [ ]:
answer_relevance = provider.relevance(QUERY, response)
print(f"Answer Relevance: {answer_relevance:.4f}  (0 = irrelevant, 1 = fully relevant)")

## 7. Context Relevance
Is each retrieved chunk relevant to the query?

In [ ]:
import numpy as np

ctx_scores = []
for i, ctx in enumerate(contexts):
    score = provider.context_relevance(QUERY, ctx)
    ctx_scores.append(score)
    print(f"  Chunk {i+1}: {score:.4f}")

context_relevance = float(np.mean(ctx_scores))
print(f"\nAverage Context Relevance: {context_relevance:.4f}")

## 8. Groundedness
Is the response supported by (grounded in) the retrieved context?

In [ ]:
groundedness_result = provider.groundedness_measure_with_cot_reasons(context_str, response)

if isinstance(groundedness_result, tuple):
    groundedness, reasons = groundedness_result
else:
    groundedness = groundedness_result
    reasons = {}

print(f"Groundedness: {float(groundedness):.4f}")
print("\nReasons:", reasons)

## 9. RAG Triad Summary

In [ ]:
ar = round(float(answer_relevance), 4)
cr = round(float(context_relevance), 4)
g  = round(float(groundedness), 4)
avg = round((ar + cr + g) / 3, 4)

print("─" * 35)
print(f"  Answer Relevance   : {ar}")
print(f"  Context Relevance  : {cr}")
print(f"  Groundedness       : {g}")
print("─" * 35)
print(f"  RAG Triad Average  : {avg}")
print("─" * 35)

## 10. Batch evaluate multiple queries

In [ ]:
test_queries = [
    "What do polar bears eat?",
    "How do polar bears stay warm?",
    "Why are polar bears endangered?",
]

results = []
for q in test_queries:
    docs = retriever.invoke(q)
    ctxs = [d.page_content for d in docs]
    ctx_combined = "\n".join(ctxs)
    res = llm.generate([f"Use the following information:\n\n{ctx_combined}\n\nQuestion: {q}"])
    ans = res.generations[0][0].text.strip()

    ar_score  = float(provider.relevance(q, ans))
    cr_scores = [float(provider.context_relevance(q, c)) for c in ctxs]
    cr_score  = float(np.mean(cr_scores))
    g_res     = provider.groundedness_measure_with_cot_reasons(ctx_combined, ans)
    g_score   = float(g_res[0] if isinstance(g_res, tuple) else g_res)

    results.append({
        "query": q,
        "answer_relevance": round(ar_score, 4),
        "context_relevance": round(cr_score, 4),
        "groundedness": round(g_score, 4),
        "avg": round((ar_score + cr_score + g_score) / 3, 4),
    })
    print(f"✓ {q}")

results

## 11. Visualise results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(results)
df = df.set_index("query")

ax = df[["answer_relevance", "context_relevance", "groundedness", "avg"]].plot(
    kind="bar", figsize=(12, 5), ylim=(0, 1), rot=20
)
ax.set_title("RAG Triad Scores per Query")
ax.set_ylabel("Score (0–1)")
ax.axhline(0.7, color="red", linestyle="--", linewidth=0.8, label="threshold 0.7")
ax.legend()
plt.tight_layout()
plt.show()

print("\nMean scores:")
print(df.mean().round(4))